In [25]:
import pathlib
import pandas as pd
import folium
from folium.plugins import MarkerCluster
import numpy as np
import os
os.makedirs("map", exist_ok=True)

In [26]:
# ── 1. Load your enriched data ────────────────────────────────────────────────
data_path = pathlib.Path(".")
data = pd.read_csv(data_path / "../data_dump/listings_with_avg_income.csv")


In [27]:
# ── 2. Build station-level summary (one row per station) ─────────────────────
stations_df = (
    data.dropna(subset=["latitude", "longitude", "station_avg_price_per_sqm"])
    .groupby("nearest_station")
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean"),
        station_avg_price_per_sqm=("station_avg_price_per_sqm", "first"),
        station_listing_count=("station_listing_count", "first"),
        station_avg_price_man_yen=("station_avg_price_man_yen", "first"),
    )
    .reset_index()
)

print(f"Stations with coordinates: {len(stations_df)}")  # 👈 add this to confirm


Stations with coordinates: 392


In [28]:
# ── 3. Color scale: green → yellow → red ─────────────────────────────────────
def price_to_color(value, vmin=42, vmax=715):
    t = np.clip((value - vmin) / (vmax - vmin), 0, 1)
    if t < 0.5:
        r = int(255 * (t * 2))
        g = 200
    else:
        r = 220
        g = int(200 * (1 - (t - 0.5) * 2))
    return f"#{r:02x}{g:02x}30"

In [29]:
# ── 4. Build map ──────────────────────────────────────────────────────────────
m = folium.Map(location=[35.6762, 139.6503], zoom_start=11)

for _, row in stations_df.iterrows():
    color = price_to_color(row["station_avg_price_per_sqm"])
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=8 + row["station_listing_count"] * 0.3,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=folium.Popup(
            f"""
            <b>{row['nearest_station']}</b><br>
            Avg price/sqm: {row['station_avg_price_per_sqm']:.1f} 万円<br>
            Avg listing price: {row['station_avg_price_man_yen']:.0f} 万円<br>
            Listings: {int(row['station_listing_count'])}
            """,
            max_width=200
        )
    ).add_to(m)

print(f"Markers added: {len(stations_df)}")  # 👈 confirm markers were added
m

Markers added: 392
